# PS16 — Restaurant Orders: Apriori Association Rule Mining

**Aim:** Find frequently purchased items using minimum support **30%** and generate association rules with minimum confidence **50%**.

**Dataset:** A small, clearly labeled example transaction CSV included with this project. It is suitable for demonstrating the algorithm, not evidence of actual customer demand.

## 1. Import libraries and locate the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

def find_csv(filename):
    choices = [
        Path.cwd() / filename,              # if notebooks + CSVs are together
        Path.cwd() / 'data' / filename,     # if CSVs are in data/
        Path.cwd().parent / filename,
        Path.cwd().parent / 'data' / filename,  # if notebook is inside notebooks/
    ]
    for candidate in choices:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f'{filename} not found. Upload the supplied CSV with the notebook '
        'or put it in a data/ folder. Checked: ' + str([str(x) for x in choices])
    )


## 2. Load and examine the transactions

Each CSV row is one item in a transaction. Several rows may share the same transaction ID.

In [ ]:
df = pd.read_csv(find_csv('restaurant_transactions.csv'))
print('Dataset shape:', df.shape)
print('Missing values:\n', df.isna().sum())
display(df.head(12))

## 3. Clean data and convert into baskets

Remove incomplete or duplicate transaction-item pairs; group items by transaction ID.

In [ ]:
df = df.dropna(subset=['transaction_id', 'item']).copy()
df['item'] = df['item'].astype(str).str.strip()
df = df[df['item'] != ''].drop_duplicates(['transaction_id', 'item'])
transactions = df.groupby('transaction_id', sort=True)['item'].apply(list).tolist()
print('Number of baskets:', len(transactions))
for i, basket in enumerate(transactions[:5], start=1):
    print(f'Basket {i}: {basket}')
assert len(transactions) > 0, 'No usable transactions' 

## 4. Apply Apriori

Use `TransactionEncoder` and `apriori()` from **mlxtend** if available. The included fallback implements the Apriori subset-pruning property without third-party mining libraries.

In [ ]:
# Use mlxtend when installed; otherwise use an offline implementation
# of Apriori's candidate generation + downward-closure pruning rule.
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules
    HAVE_MLXTEND = True
    print('Apriori backend: mlxtend')
except ImportError:
    HAVE_MLXTEND = False
    print('Apriori backend: built-in Python fallback (mlxtend not installed)')

from itertools import combinations

def fallback_apriori(transactions, min_support):
    baskets = [set(t) for t in transactions]
    n = len(baskets)
    unique_items = sorted(set().union(*baskets))
    previous_level = set()
    result = []

    for size in range(1, max(map(len, baskets)) + 1):
        candidates = [frozenset(items) for items in combinations(unique_items, size)]
        if size > 1:
            # Apriori property: every (size-1)-subset must be frequent.
            candidates = [candidate for candidate in candidates
                          if all(frozenset(subset) in previous_level
                                 for subset in combinations(candidate, size - 1))]
        current_level = set()
        for candidate in candidates:
            support = sum(candidate.issubset(basket) for basket in baskets) / n
            if support + 1e-12 >= min_support:
                current_level.add(candidate)
                result.append({'support': support, 'itemsets': candidate})
        if not current_level:
            break
        previous_level = current_level
    return pd.DataFrame(result, columns=['support', 'itemsets'])


def fallback_association_rules(frequent, min_confidence):
    supports = {frozenset(row.itemsets): float(row.support)
                for row in frequent.itertuples(index=False)}
    records = []
    for itemset, itemset_support in supports.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for left in combinations(sorted(itemset), size):
                antecedent = frozenset(left)
                consequent = itemset - antecedent
                confidence = itemset_support / supports[antecedent]
                if confidence + 1e-12 >= min_confidence:
                    records.append({
                        'antecedents': antecedent,
                        'consequents': consequent,
                        'support': itemset_support,
                        'confidence': confidence,
                        'lift': confidence / supports[consequent]
                    })
    return pd.DataFrame(records, columns=[
        'antecedents', 'consequents', 'support', 'confidence', 'lift'
    ])


In [ ]:
MIN_SUPPORT = 0.3
MIN_CONFIDENCE = 0.5

if HAVE_MLXTEND:
    te = TransactionEncoder()
    encoded_array = te.fit(transactions).transform(transactions)
    basket_df = pd.DataFrame(encoded_array, columns=te.columns_)
    frequent_itemsets = apriori(basket_df, min_support=MIN_SUPPORT, use_colnames=True)
else:
    # Equivalent transaction/item indicator table for display.
    unique_items = sorted(set(item for basket in transactions for item in basket))
    basket_df = pd.DataFrame([
        {item: item in basket for item in unique_items} for basket in transactions
    ])
    frequent_itemsets = fallback_apriori(transactions, MIN_SUPPORT)

print('One-hot encoded transactions:')
display(basket_df.head())
print('Frequent itemsets:', len(frequent_itemsets))
if frequent_itemsets.empty:
    print('No itemsets meet the support threshold.')
else:
    view = frequent_itemsets.copy()
    view['itemsets'] = view['itemsets'].apply(lambda items: ', '.join(sorted(items)))
    display(view.sort_values('support', ascending=False).reset_index(drop=True))

## 5. Generate association rules

**Support:** fraction of baskets containing both sides. **Confidence:** P(consequent | antecedent). **Lift:** confidence divided by consequent support; lift above 1 suggests positive association.

In [ ]:
if frequent_itemsets.empty:
    rules = pd.DataFrame(columns=['antecedents', 'consequents', 'support', 'confidence', 'lift'])
elif HAVE_MLXTEND:
    # num_itemsets is required by some mlxtend versions; accepted by newer ones.
    rules = association_rules(
        frequent_itemsets, num_itemsets=len(basket_df),
        metric='confidence', min_threshold=MIN_CONFIDENCE
    )
else:
    rules = fallback_association_rules(frequent_itemsets, MIN_CONFIDENCE)

if rules.empty:
    print('No association rules meet the confidence threshold.')
else:
    display_rules = rules[['antecedents','consequents','support','confidence','lift']].copy()
    for name in ['antecedents', 'consequents']:
        display_rules[name] = display_rules[name].apply(lambda items: ', '.join(sorted(items)))
    display_rules = display_rules.sort_values(['confidence','lift'], ascending=False)
    display(display_rules.reset_index(drop=True).round(3))

## 6. Plot frequent itemset support and print a recommendation

In [ ]:
if not frequent_itemsets.empty:
    plot_df = frequent_itemsets.copy()
    plot_df['items'] = plot_df['itemsets'].apply(lambda s: ', '.join(sorted(s)))
    plot_df = plot_df.sort_values('support').tail(10)
    ax = plot_df.plot.barh(x='items', y='support', legend=False, figsize=(9, 4))
    ax.set_xlabel('Support (fraction of transactions)')
    ax.set_ylabel('Frequent itemset')
    plt.tight_layout()
    plt.show()

if not rules.empty:
    best = display_rules.iloc[0]
    print(f"Example recommendation: customers who buy [{best['antecedents']}] "
          f"may also buy [{best['consequents']}].")
    print(f"Confidence = {best['confidence']:.1%}; Lift = {best['lift']:.2f}")

## 7. Conclusion and viva answers

- **Apriori** discovers frequent itemsets level by level and prunes candidates whose subsets are infrequent.
- **Support threshold:** 30% of all baskets; **confidence threshold:** 50% for an association rule.
- **One-hot encoding:** `True` means an item is present in the transaction.
- **Association rule** `A → B`: if a basket includes A, it may also contain B; this is not causation.
- **Lift** above 1 indicates co-occurrence greater than chance given marginal supports.
- **Suggested use:** create meal combos for frequently paired items.

**Dataset limitation:** Results depend on the supplied example transactions and should not be interpreted as real-world customer behaviour.